# LeetCode 2833 — Furthest Point From Origin bằng Python

Link đề: https://leetcode.com/problems/furthest-point-from-origin/ · Độ khó: **Easy** · Chủ đề: String, Counting, Greedy

Skeleton LeetCode:

```python
class Solution:
    def furthestDistanceFromOrigin(self, moves: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bạn đứng ở vị trí `0` trên trục số, chuỗi `moves` chứa `'L'` (sang trái), `'R'` (sang phải) và `'_'` (**được chọn** trái hoặc phải); tìm **khoảng cách xa nhất** tới gốc có thể đạt được sau `n` bước.
2. Biết **tự tính tay**: đếm số `L`, `R`, `_`, rồi suy ra "dồn hết `_` về cùng một phía".
3. Học các công cụ Python: `str.count`, giá trị tuyệt đối `abs`, vòng `for` duyệt chuỗi, `itertools.product` để thử mọi lựa chọn, `set` các vị trí có thể đạt, `max` / `min`.
4. Đi từ **Brute Force: thử mọi cách chọn `_` (`O(2^k)`)** → **tập các vị trí có thể đạt (`O(n²)`)** → **chỉ thử hai trường hợp cực đoan (`O(n)`)** → **công thức `|L − R| + _`**.
5. Chứng minh vì sao dồn tất cả `_` về cùng phía là tối ưu, và tránh lỗi kinh điển: quên `abs`, lấy `max(L, R)`, cố gắng chia `_` cho cả hai phía.

## 2. Đề bài & ví dụ

Bạn được cho một chuỗi `moves` độ dài `n`, chỉ gồm các ký tự `'L'`, `'R'` và `'_'`. Chuỗi biểu diễn các **bước đi** của bạn trên một **trục số** bắt đầu từ điểm `0`. Ở bước thứ `i`, bạn có thể di chuyển như sau:

- Nếu `moves[i] = 'L'` hoặc `moves[i] = '_'`, bạn có thể đi **sang trái** một đơn vị.
- Nếu `moves[i] = 'R'` hoặc `moves[i] = '_'`, bạn có thể đi **sang phải** một đơn vị.

Trả về **khoảng cách lớn nhất** từ điểm gốc `0` mà bạn có thể đạt được **sau `n` bước**.

Nghĩa là `'L'` bắt buộc đi trái, `'R'` bắt buộc đi phải, còn `'_'` bạn **tự chọn** trái hoặc phải.

| Ví dụ | `moves` | Output | Giải thích |
|---|---|---|---|
| 1 | `"L_RL__R"` | `3` | Điểm xa nhất là `-3`, đạt được bằng dãy bước `"LLRLLLR"`. |
| 2 | `"_R__LL_"` | `5` | Điểm xa nhất là `-5`, đạt được bằng dãy bước `"LRLLLLL"`. |
| 3 | `"_______"` | `7` | Điểm xa nhất là `7`, đạt được bằng dãy bước `"RRRRRRR"`. |

Constraints:

- `1 <= moves.length == n <= 50`
- `moves` chỉ gồm các ký tự `'L'`, `'R'` và `'_'`.

In [1]:
# Chạy thử "bằng tay" ví dụ 3: "_______" toàn lựa chọn
moves = "_______"
print("đi toàn R:", moves.count("_"), "-> vị trí", moves.count("_"))
print("đi toàn L:", moves.count("_"), "-> vị trí", -moves.count("_"))

đi toàn R: 7 -> vị trí 7
đi toàn L: 7 -> vị trí -7


## 3. Kiến thức nền

### 3.1. Vị trí trên trục số và khoảng cách `abs`

Mỗi bước `R` cộng `+1`, mỗi bước `L` cộng `−1` vào **vị trí** `pos`. **Khoảng cách tới gốc** là `abs(pos)` (giá trị tuyệt đối), không âm: ở `−3` hay `+3` đều cách gốc `3`.

In [2]:
pos = 0
for ch in "RRL":
    pos += 1 if ch == "R" else -1
    print(ch, "-> pos =", pos)
print("khoảng cách:", abs(pos), "| abs(-3) =", abs(-3))

R -> pos = 1
R -> pos = 2
L -> pos = 1
khoảng cách: 1 | abs(-3) = 3


### 3.2. Đếm ký tự bằng `str.count`

`s.count("L")` trả về số lần ký tự (hoặc chuỗi con) `"L"` xuất hiện trong `s`. Mỗi lần gọi duyệt cả chuỗi `O(n)`.

In [3]:
moves = "L_RL__R"
print(moves.count("L"), moves.count("R"), moves.count("_"))    # 2 2 3
print(len(moves), moves.count("L") + moves.count("R") + moves.count("_"))   # 7 7

2 2 3
7 7


### 3.3. Duyệt chuỗi bằng `for ch in s` và đếm thủ công

Cách khác với `count`: một vòng `for` và ba biến đếm `l`, `r`, `u`. Một lượt duyệt `O(n)`.

In [4]:
l = r = u = 0
for ch in "L_RL__R":
    if ch == "L":
        l += 1
    elif ch == "R":
        r += 1
    else:
        u += 1
print(l, r, u)       # 2 2 3

2 2 3


### 3.4. `itertools.product` — thử mọi tổ hợp lựa chọn

`product("LR", repeat=k)` sinh mọi dãy độ dài `k` gồm `L` hoặc `R` (có `2^k` dãy). Ta dùng để thử mọi cách "điền" vào các ô `_`.

In [5]:
from itertools import product

print(list(product("LR", repeat=2)))     # [('L', 'L'), ('L', 'R'), ('R', 'L'), ('R', 'R')]
print(len(list(product("LR", repeat=5))))   # 32

[('L', 'L'), ('L', 'R'), ('R', 'L'), ('R', 'R')]
32


### 3.5. `set` các vị trí có thể đạt

`set` là tập hợp không trùng phần tử. Ta có thể giữ **tập mọi vị trí có thể** sau mỗi bước: bước `L` thì dịch mọi vị trí `−1`, bước `R` thì `+1`, bước `_` thì hợp của cả hai.

In [6]:
cur = {0}
cur = {p - 1 for p in cur} | {p + 1 for p in cur}    # bước '_'
print(cur)                                           # {-1, 1}
cur = {p + 1 for p in cur}                           # bước 'R'
print(cur)                                           # {0, 2}

{1, -1}
{0, 2}


### 3.6. `max` / `min` trên nhiều giá trị

`max(a, b)` lấy số lớn hơn; `max(iterable)` lấy lớn nhất. Ta cần `max(abs(p) for p in vị_trí)`.

In [7]:
print(max(3, -1), max([abs(-3), abs(2), abs(0)]))   # 3 3

3 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1 `"L_RL__R"`: có `L = 2`, `R = 2`, `_ = 3`.

Các bước bắt buộc: hai `L` đẩy sang trái `−2`, hai `R` đẩy sang phải `+2`. **Khi chỉ có bước bắt buộc, vị trí là `+2 − 2 = 0`.** Còn 3 ô `_` để tự chọn.

- Nếu cả 3 ô `_` đều đi trái: vị trí `0 − 3 = −3`, khoảng cách `3`.
- Nếu cả 3 ô `_` đều đi phải: vị trí `0 + 3 = +3`, khoảng cách `3`.
- Nếu 2 ô trái, 1 ô phải: vị trí `−1`, khoảng cách `1` (kém hơn).

Vậy **tốt nhất là dồn hết `_` về cùng một phía**, phía mà bước bắt buộc đã "nghiêng" về (nếu có). Ở ví dụ 2 `"_R__LL_"`: `L = 2`, `R = 1`, `_ = 4`. Bước bắt buộc: `1 − 2 = −1` (nghiêng trái 1). Dồn 4 ô `_` sang **trái**: `−1 − 4 = −5`, khoảng cách `5`.

**Công thức:** khoảng cách tối đa = `|L − R| + _`. Với ví dụ 2: `|2 − 1| + 4 = 5` ✓. Với ví dụ 1: `|2 − 2| + 3 = 3` ✓.

In [8]:
for moves in ["L_RL__R", "_R__LL_", "_______"]:
    L, R, U = moves.count("L"), moves.count("R"), moves.count("_")
    print(f"{moves}: L={L}, R={R}, _={U} -> |L-R|+_ = {abs(L - R)} + {U} = {abs(L - R) + U}")

L_RL__R: L=2, R=2, _=3 -> |L-R|+_ = 0 + 3 = 3
_R__LL_: L=2, R=1, _=4 -> |L-R|+_ = 1 + 4 = 5
_______: L=0, R=0, _=7 -> |L-R|+_ = 0 + 7 = 7


## 5. Cách 1 — Brute Force / baseline: thử mọi cách chọn `_`

**Ý tưởng:** có `k` ô `_`, mỗi ô chọn `L` hoặc `R`: `2^k` cách. Với mỗi cách, ta "điền" vào chuỗi để thành dãy bước cụ thể rồi tính vị trí cuối; lấy `max` của `abs(vị_trí)`.

Cách này dịch thẳng đề, rất tốt làm **oracle** cho `k` nhỏ (ví dụ `k <= 12`).

In [9]:
from itertools import product
from typing import List

def furthest_bruteforce(moves: str) -> int:
    idx = [i for i, ch in enumerate(moves) if ch == "_"]     # vị trí các ô tự chọn
    best = 0
    for choice in product("LR", repeat=len(idx)):             # mọi cách điền
        arr = list(moves)
        for i, c in zip(idx, choice):
            arr[i] = c
        pos = sum(1 if ch == "R" else -1 for ch in arr)
        best = max(best, abs(pos))
    return best

In [10]:
print(furthest_bruteforce("L_RL__R"))     # 3
print(furthest_bruteforce("_R__LL_"))     # 5
print(furthest_bruteforce("_______"))     # 7
print(furthest_bruteforce("LLR"))         # 1  (không có ô tự chọn: product rỗng cho đúng 1 dãy)

3
5
7
1


**Độ phức tạp:** `k` là số ô `_`. Có `2^k` cách điền, mỗi cách tính vị trí `O(n)` ⇒ Time `O(2^k · n)`. Với `k = 50` thì `2^50 ≈ 10^15` — không thể chạy. Space `O(n)`.

**Nút thắt:** số cách tăng theo luỹ thừa. Ta cần nhận ra nhiều cách cho **cùng một vị trí**: chỉ cần biết *tập vị trí đạt được*, hoặc tốt hơn, chỉ biết vị trí xa nhất.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — chỉ vị trí cuối quan trọng.** Vị trí cuối = (số `R` + số ô `_` chọn phải) − (số `L` + số ô `_` chọn trái). Nhiều cách điền cho cùng vị trí. Nếu có `a` ô `_` chọn phải và `k − a` ô chọn trái, vị trí cuối là `R + a − L − (k − a) = (R − L) + 2a − k`. Chỉ phụ thuộc `a`, không phụ thuộc **ô nào** được chọn.

**Đột phá 2 — hàm tuyến tính theo `a`, cực đại ở biên.** Vị trí `p(a) = (R − L − k) + 2a` tăng theo `a`, với `a ∈ [0, k]`. Vị trí **lớn nhất** là `a = k` (mọi `_` đi phải): `R − L + k`. Vị trí **nhỏ nhất** là `a = 0` (mọi `_` đi trái): `R − L − k`. Mọi vị trí khác nằm giữa hai số này.

**Đột phá 3 — khoảng cách xa nhất** là `max(|R − L + k|, |R − L − k|)`; vì `abs` của hàm ở hai đầu mút của một đoạn thì max của `|x|` trên một đoạn đạt ở **đầu mút**. Hơn nữa `max(|d + k|, |d − k|) = |d| + k` với `k >= 0`, nên đáp án = `|R − L| + k`.

Vậy chỉ cần thử hai trường hợp cực đoan, hoặc tính thẳng bằng công thức.

In [11]:
moves = "L_RL__R"
L, R, k = moves.count("L"), moves.count("R"), moves.count("_")
d = R - L
print("mọi _ sang phải: vị trí", d + k)
print("mọi _ sang trái: vị trí", d - k)
print("khoảng cách xa nhất:", max(abs(d + k), abs(d - k)), "=", abs(d) + k)

mọi _ sang phải: vị trí 3
mọi _ sang trái: vị trí -3
khoảng cách xa nhất: 3 = 3


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa.

### Cách 2 — Tập các vị trí có thể đạt sau mỗi bước (DP trên tập)

**Từng đoạn:**

- `cur = {0}`: lúc đầu chỉ có thể ở `0`.
- Với mỗi bước: `L` ⇒ mọi vị trí `−1`; `R` ⇒ mọi vị trí `+1`; `_` ⇒ hợp của hai dịch chuyển.
- Cuối: `max(abs(p) for p in cur)`.

Các vị trí có thể đạt sau `i` bước nằm trong `[−i, i]`, tối đa `i + 1` giá trị khác nhau (cùng tính chẵn lẻ), nên tập không bùng nổ như `2^k`.

In [12]:
def furthest_set(moves: str) -> int:
    cur = {0}
    for ch in moves:
        if ch == "L":
            cur = {p - 1 for p in cur}
        elif ch == "R":
            cur = {p + 1 for p in cur}
        else:
            cur = {p - 1 for p in cur} | {p + 1 for p in cur}
    return max(abs(p) for p in cur)

In [13]:
print(furthest_set("L_RL__R"))     # 3
print(furthest_set("_R__LL_"))     # 5
print(furthest_set("_______"))     # 7

3
5
7


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def furthestDistanceFromOrigin(self, moves: str) -> int:
        cur = {0}
        for ch in moves:
            if ch == "L":
                cur = {p - 1 for p in cur}
            elif ch == "R":
                cur = {p + 1 for p in cur}
            else:
                cur = {p - 1 for p in cur} | {p + 1 for p in cur}
        return max(abs(p) for p in cur)
```

**Độ phức tạp:** mỗi bước xử lý tập kích thước `O(n)` ⇒ Time `O(n²)`, Space `O(n)`.

### Cách 3 — Hai kịch bản cực đoan (greedy)

**Từng đoạn:**

- Kịch bản 1: coi mọi `'_'` là `'L'`, mô phỏng vị trí.
- Kịch bản 2: coi mọi `'_'` là `'R'`, mô phỏng vị trí.
- Đáp án = `max(abs(pos1), abs(pos2))`.

In [14]:
def furthest_two_scenarios(moves: str) -> int:
    pos_left = 0     # mọi '_' đi trái
    pos_right = 0    # mọi '_' đi phải
    for ch in moves:
        if ch == "L":
            pos_left -= 1
            pos_right -= 1
        elif ch == "R":
            pos_left += 1
            pos_right += 1
        else:
            pos_left -= 1
            pos_right += 1
    return max(abs(pos_left), abs(pos_right))

In [15]:
print(furthest_two_scenarios("L_RL__R"))     # 3
print(furthest_two_scenarios("_R__LL_"))     # 5
print(furthest_two_scenarios("_______"))     # 7

3
5
7


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def furthestDistanceFromOrigin(self, moves: str) -> int:
        pos_left = pos_right = 0
        for ch in moves:
            if ch == "L":
                pos_left -= 1
                pos_right -= 1
            elif ch == "R":
                pos_left += 1
                pos_right += 1
            else:
                pos_left -= 1
                pos_right += 1
        return max(abs(pos_left), abs(pos_right))
```

**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Công thức `|L − R| + _` (dùng để nộp)

**Từng đoạn:** `L = moves.count('L')`, `R = moves.count('R')`, `U = moves.count('_')`. Đáp án `abs(L − R) + U`.

In [16]:
def furthest_formula(moves: str) -> int:
    return abs(moves.count("L") - moves.count("R")) + moves.count("_")

In [17]:
print(furthest_formula("L_RL__R"))     # 3
print(furthest_formula("_R__LL_"))     # 5
print(furthest_formula("_______"))     # 7

3
5
7


**Độ phức tạp:** ba lần `count`, mỗi lần `O(n)` ⇒ Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 (hai kịch bản) trên `moves = "_R__LL_"`. Ta theo dõi `pos_left` (mọi `_` đi trái) và `pos_right` (mọi `_` đi phải).

| bước | ký tự | `pos_left` sau | `pos_right` sau |
|---|---|---|---|
| 0 | — | 0 | 0 |
| 1 | `_` | −1 | +1 |
| 2 | `R` | 0 | +2 |
| 3 | `_` | −1 | +3 |
| 4 | `_` | −2 | +4 |
| 5 | `L` | −3 | +3 |
| 6 | `L` | −4 | +2 |
| 7 | `_` | **−5** | +3 |

Khoảng cách: `max(|−5|, |3|) = 5` ✓.

In [18]:
def trace_two_scenarios(moves):
    pl = pr = 0
    print(f"bắt đầu: pos_left={pl}, pos_right={pr}")
    for i, ch in enumerate(moves, 1):
        if ch == "L":
            pl, pr = pl - 1, pr - 1
        elif ch == "R":
            pl, pr = pl + 1, pr + 1
        else:
            pl, pr = pl - 1, pr + 1
        print(f"bước {i}: ch={ch!r} -> pos_left={pl:3}, pos_right={pr:3}")
    ans = max(abs(pl), abs(pr))
    print(f"đáp án = max(|{pl}|, |{pr}|) = {ans}")
    return ans

trace_two_scenarios("_R__LL_")

bắt đầu: pos_left=0, pos_right=0
bước 1: ch='_' -> pos_left= -1, pos_right=  1
bước 2: ch='R' -> pos_left=  0, pos_right=  2
bước 3: ch='_' -> pos_left= -1, pos_right=  3
bước 4: ch='_' -> pos_left= -2, pos_right=  4
bước 5: ch='L' -> pos_left= -3, pos_right=  3
bước 6: ch='L' -> pos_left= -4, pos_right=  2
bước 7: ch='_' -> pos_left= -5, pos_right=  3
đáp án = max(|-5|, |3|) = 5


5

In [19]:
# Trace Cách 2 (tập vị trí) trên ví dụ nhỏ để thấy tập lớn dần
def trace_set(moves):
    cur = {0}
    print("bắt đầu:", sorted(cur))
    for i, ch in enumerate(moves, 1):
        if ch == "L":
            cur = {p - 1 for p in cur}
        elif ch == "R":
            cur = {p + 1 for p in cur}
        else:
            cur = {p - 1 for p in cur} | {p + 1 for p in cur}
        print(f"bước {i}: ch={ch!r} -> {sorted(cur)}")
    print("max |p| =", max(abs(p) for p in cur))

trace_set("L_R_")

bắt đầu: [0]
bước 1: ch='L' -> [-1]
bước 2: ch='_' -> [-2, 0]
bước 3: ch='R' -> [-1, 1]
bước 4: ch='_' -> [-2, 0, 2]
max |p| = 2


## 9. Code nộp LeetCode

Dùng Cách 4 (công thức): ngắn, `O(n)`, dễ chứng minh.

In [20]:
class Solution:
    def furthestDistanceFromOrigin(self, moves: str) -> int:
        return abs(moves.count("L") - moves.count("R")) + moves.count("_")

In [21]:
print(Solution().furthestDistanceFromOrigin("L_RL__R"))     # 3
print(Solution().furthestDistanceFromOrigin("_R__LL_"))     # 5
print(Solution().furthestDistanceFromOrigin("_______"))     # 7

3
5
7


## 10. Vì sao thuật toán đúng

Đặt `L, R, k` là số ký tự `'L'`, `'R'`, `'_'`; `d = R − L`.

**Bước 1 — vị trí cuối.** Gọi `a` là số ô `_` chọn **phải** (`0 <= a <= k`), thì có `k − a` ô chọn trái. Vị trí cuối `p(a) = R + a − L − (k − a) = d − k + 2a`. Mọi cách chọn cho cùng `a` cho cùng vị trí.

**Bước 2 — tập vị trí có thể** là `{d − k + 2a : a = 0..k}`, tức các số từ `d − k` tới `d + k` cách nhau `2`. Hai đầu mút là `d − k` và `d + k`.

**Bước 3 — `|p|` lớn nhất ở đầu mút.** Hàm `|x|` là **lồi**, nên trên một đoạn `[lo, hi]` nó đạt cực đại tại **đầu mút** (`lo` hoặc `hi`). Vậy khoảng cách lớn nhất = `max(|d − k|, |d + k|)`.

**Bước 4 — rút gọn.** Với `k >= 0`: nếu `d >= 0` thì `|d + k| = d + k` và `|d − k| <= d + k` ⇒ max = `d + k = |d| + k`. Nếu `d < 0`, đối xứng: `|d − k| = |d| + k`. Vậy đáp án `|d| + k = |R − L| + k = |L − R| + k`.

**Đối chiếu:** Cách 3 tính đúng hai đầu mút (`pos_left = d − k`, `pos_right = d + k`); Cách 2 liệt kê toàn bộ tập vị trí; Cách 1 thử mọi dãy. Cả ba trùng công thức (sẽ kiểm bằng test).

## 11. Phân tích Time / Space Complexity

`n = len(moves)`, `k` = số ô `_`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi cách chọn `_` | `O(2^k · n)` | `O(n)` | chỉ dùng cho `k` nhỏ, làm oracle |
| 2. Tập vị trí | `O(n²)` | `O(n)` | DP trên tập |
| 3. Hai kịch bản cực đoan | `O(n)` | `O(1)` | một lượt duyệt |
| 4. Công thức `|L − R| + _` | `O(n)` | `O(1)` | **code nộp** |

Với `n <= 50`, các cách 2–4 tức thì; cách 1 chỉ chạy được khi `k` nhỏ (khoảng `<= 14`).

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: không có `_`; toàn `_`; toàn `L`; toàn `R`; `L` và `R` bằng nhau; một ký tự; `L` và `R` xen kẽ có `_`.

In [22]:
impls = {
    "set": furthest_set,
    "two_scenarios": furthest_two_scenarios,
    "formula": furthest_formula,
    "Solution": Solution().furthestDistanceFromOrigin,
}
cases = [
    ("ví dụ 1", "L_RL__R", 3),
    ("ví dụ 2", "_R__LL_", 5),
    ("ví dụ 3", "_______", 7),
    ("một ký tự L", "L", 1),
    ("một ký tự R", "R", 1),
    ("một ký tự _", "_", 1),
    ("không có _", "LLRLR", 1),
    ("L và R cân bằng, không _", "LRLR", 0),
    ("toàn L", "LLLL", 4),
    ("toàn R", "RRR", 3),
    ("cân bằng nhưng có _", "LR_", 1),
]
for name, moves, exp in cases:
    assert furthest_bruteforce(moves) == exp, ("bruteforce", name)
    for k, f in impls.items():
        got = f(moves)
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 3
PASS ví dụ 2 -> 5
PASS ví dụ 3 -> 7
PASS một ký tự L -> 1
PASS một ký tự R -> 1
PASS một ký tự _ -> 1
PASS không có _ -> 1
PASS L và R cân bằng, không _ -> 0
PASS toàn L -> 4
PASS toàn R -> 3
PASS cân bằng nhưng có _ -> 1


Kiểm tay ca `"LRLR"`: hai `L`, hai `R`, không có `_` ⇒ vị trí cuối `0` ⇒ khoảng cách `0` ✓. Ca `"LR_"`: `L = R = 1`, `_ = 1` ⇒ `|1 − 1| + 1 = 1` ✓.

### 12.2. So sánh các cách trên nhiều input ngẫu nhiên có seed

Input hợp lệ theo constraints: `1 <= n <= 50`, chỉ `'L'`, `'R'`, `'_'`. Brute force chỉ chạy khi `k` nhỏ (`k <= 12`), nên ta tạo hai nhóm: chuỗi có ít `_` (so với brute force) và chuỗi bất kỳ (so các cách còn lại).

In [23]:
import random
random.seed(2833)
dem_brute = 0
for _ in range(2000):
    n = random.randint(1, 50)
    weights = random.choice([(1, 1, 1), (1, 1, 6), (3, 1, 1), (1, 3, 1), (5, 5, 1)])
    moves = "".join(random.choices("LR_", weights=weights, k=n))
    exp = furthest_formula(moves)
    for k, f in impls.items():
        assert f(moves) == exp, (moves, k)
    if moves.count("_") <= 12:
        assert furthest_bruteforce(moves) == exp, moves
        dem_brute += 1
print("2000 chuỗi ngẫu nhiên khớp; trong đó", dem_brute, "chuỗi được đối chiếu thêm với brute force")

2000 chuỗi ngẫu nhiên khớp; trong đó 1567 chuỗi được đối chiếu thêm với brute force


### 12.3. Kiểm tính chất

- Đáp án nằm trong `[0, n]`.
- Đáp án `>= k` (số `_`), vì ta luôn có thể dồn mọi `_` một phía rồi cộng với `|R − L|`.
- Đổi chỗ tất cả `L` ↔ `R` (đối xứng gương) không đổi đáp án.
- Hoán vị ký tự trong `moves` không đổi đáp án (vì chỉ phụ thuộc số lượng).

In [24]:
random.seed(28330)
for _ in range(500):
    n = random.randint(1, 50)
    moves = "".join(random.choice("LR_") for _ in range(n))
    r = Solution().furthestDistanceFromOrigin(moves)
    assert 0 <= r <= n
    assert r >= moves.count("_")
    mirror = moves.translate(str.maketrans("LR", "RL"))
    assert Solution().furthestDistanceFromOrigin(mirror) == r
    arr = list(moves)
    random.shuffle(arr)
    assert Solution().furthestDistanceFromOrigin("".join(arr)) == r
print("tính chất đúng")

tính chất đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `abs` (trả vị trí thay vì khoảng cách)

In [25]:
def sai_quen_abs(moves):
    return moves.count("R") - moves.count("L") + moves.count("_")     # chỉ đúng khi nghiêng phải

print("sai :", sai_quen_abs("_R__LL_"))                       # 3 (sai)
print("đúng:", Solution().furthestDistanceFromOrigin("_R__LL_"))   # 5

sai : 3
đúng: 5


**Vì sao sai:** nếu bước bắt buộc nghiêng **trái** (`L > R`) thì nên dồn `_` sang **trái**, và đáp án là `|R − L| + _`, không phải `R − L + _`. Cần `abs`.

### Lỗi 2: Lấy `max(L, R)` rồi cộng `_` hoặc quên trừ

In [26]:
def sai_max_lr(moves):
    return max(moves.count("L"), moves.count("R")) + moves.count("_")

print("sai :", sai_max_lr("L_RL__R"))                          # 5 (sai)
print("đúng:", Solution().furthestDistanceFromOrigin("L_RL__R"))    # 3

sai : 5
đúng: 3


**Vì sao sai:** các bước `L` và `R` bắt buộc **triệt tiêu** nhau; chỉ phần dư `|L − R|` mới góp vào khoảng cách. `max(L, R)` bỏ qua việc triệt tiêu.

### Lỗi 3: Chia `_` cho cả hai phía

In [27]:
def sai_chia_doi(moves):
    L, R, k = moves.count("L"), moves.count("R"), moves.count("_")
    return abs(R + k // 2 - L - (k - k // 2))     # sai: chia đều _ hai phía

print("sai :", sai_chia_doi("_______"))                          # 1
print("đúng:", Solution().furthestDistanceFromOrigin("_______"))    # 7

sai : 1
đúng: 7


**Vì sao sai:** để **xa nhất** ta phải dồn mọi `_` về **cùng một phía**; chia đều làm các bước triệt tiêu nhau.

### Lỗi 4: Nghĩ `'_'` là "đứng yên"

In [28]:
def sai_dung_yen(moves):
    return abs(moves.count("L") - moves.count("R"))     # coi '_' không làm gì

print("sai :", sai_dung_yen("_______"))                          # 0
print("đúng:", Solution().furthestDistanceFromOrigin("_______"))    # 7

sai : 0
đúng: 7


**Vì sao sai:** mỗi `'_'` là **một bước đi** (trái hoặc phải), không phải đứng yên. Sau `n` bước bạn đã đi đúng `n` bước.

### Lỗi 5 (nhỏ): Đếm sai vì dùng `moves.count("_")` với ký tự lạ

Ký tự gạch dưới là `_` (underscore). Nếu gõ nhầm `-` (gạch ngang), `count` sẽ cho `0`. Hãy cẩn thận khi copy ký tự.

In [29]:
print("a_b".count("_"), "a-b".count("_"))     # 1 0

1 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có "ô được chọn tự do" mà hỏi **cực đại / cực tiểu** thường dẫn tới **greedy**: dồn lựa chọn tự do về hướng có lợi nhất.
- Khi các bước bắt buộc ngược chiều nhau, chúng **triệt tiêu**: chỉ giữ phần dư (`|L − R|`).
- Khoảng cách xa nhất = `|bắt buộc ròng| + số ô tự do`.
- Hàm `|x|` lồi ⇒ cực đại trên đoạn đạt ở **đầu mút**; chỉ cần xét hai kịch bản cực đoan.
- Có thể dùng brute force (`product`) trên chuỗi nhỏ để **kiểm** công thức bằng vét cạn.

## 15. Biến thể của bài

1. **Gần gốc nhất:** tìm khoảng cách **nhỏ nhất** tới gốc. Tập vị trí cuối là `{d − k, d − k + 2, ..., d + k}`; nếu `0` nằm trong đoạn đó thì đáp án là `0` hoặc `1` tuỳ tính chẵn lẻ của `d − k`, còn không thì là `min(|d − k|, |d + k|)`.
2. **Có thể đứng yên:** thêm ký tự `'S'` (bắt buộc đứng yên).
3. **Chiều thứ hai:** `U`, `D`, `L`, `R` trên mặt phẳng; khoảng cách Manhattan.
4. **Đếm số cách** đạt khoảng cách xa nhất: dùng tổ hợp.
5. **Đếm vị trí cuối có thể đạt:** `k + 1` vị trí `d − k, d − k + 2, ..., d + k`.
6. **Chi phí khác nhau cho mỗi bước:** thay `±1` bằng `±w[i]`; khi đó phải dùng DP trên tập vị trí.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cách chọn | `product` mọi dãy `L/R` cho `_` | `O(2^k · n)` | `O(n)` | oracle, `k` nhỏ |
| 2. Tập vị trí | DP trên tập vị trí có thể | `O(n²)` | `O(n)` | khi bước có trọng số khác nhau |
| 3. Hai kịch bản | mô phỏng hai cực đoan | `O(n)` | `O(1)` | khi chưa nhớ công thức |
| 4. Công thức | `|L − R| + _` | `O(n)` | `O(1)` | **code nộp** |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"L_R_".count("_")`, `abs(2 - 5)`, `max(abs(-4), abs(3))`, `len(list(product("LR", repeat=3)))`.

**Bài 2 (tính tay).** Với `moves = "LL__R"`: đếm `L`, `R`, `_`; vị trí xa nhất khi dồn `_` sang trái và sang phải; đáp án.

**Bài 3 (điền TODO).** Hoàn thành `furthest_todo(moves)` theo công thức.

In [30]:
def furthest_todo(moves):
    # TODO 1: L = số ký tự 'L'
    # TODO 2: R = số ký tự 'R'
    # TODO 3: U = số ký tự '_'
    # TODO 4: return abs(L - R) + U
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `furthest_two_scenarios2(moves)` mô phỏng hai kịch bản (mọi `_` là `L` rồi mọi `_` là `R`) **không nhìn** code mẫu.

**Bài 5 (biến thể / mini challenge).** (a) Viết `closest_distance(moves)`: khoảng cách **nhỏ nhất** tới gốc có thể đạt (gợi ý: tập vị trí là `{d − k, d − k + 2, ..., d + k}`). (b) Viết `reachable_positions(moves)` trả về **list** các vị trí cuối có thể đạt, đã sắp xếp tăng dần.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"L_R_".count("_") = 2`; `abs(2 − 5) = 3`; `max(4, 3) = 4`; `2^3 = 8`. Bài 2: `"LL__R"` có `L = 2`, `R = 1`, `_ = 2`.
- **Mức 2 (vừa):** Bài 3: bốn dòng đã ghi trong comment TODO. Bài 4: hai biến `pl`, `pr`; `_` làm `pl -= 1` và `pr += 1`.
- **Mức 3 (sâu):** Bài 5(a): vị trí cuối có thể là `d − k + 2a`; khoảng cách nhỏ nhất là `min(|d − k + 2a|)` trên `a = 0..k`; có thể duyệt `a`. Bài 5(b): `[d − k + 2*a for a in range(k + 1)]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert "L_R_".count("_") == 2
assert abs(2 - 5) == 3
assert max(abs(-4), abs(3)) == 4
assert len(list(product("LR", repeat=3))) == 8

# Bài 2: "LL__R": L=2, R=1, _=2
# dồn _ sang trái: pos = 1 - 2 - 2 = -3 ; dồn _ sang phải: pos = 1 - 2 + 2 = 1 -> max khoảng cách 3
assert Solution().furthestDistanceFromOrigin("LL__R") == 3

# Bài 3
def furthest_todo_ans(moves):
    L = moves.count("L")
    R = moves.count("R")
    U = moves.count("_")
    return abs(L - R) + U

assert furthest_todo_ans("L_RL__R") == 3
assert furthest_todo_ans("_______") == 7

# Bài 4
def furthest_two_scenarios2(moves):
    pl = pr = 0
    for ch in moves:
        if ch == "L":
            pl -= 1; pr -= 1
        elif ch == "R":
            pl += 1; pr += 1
        else:
            pl -= 1; pr += 1
    return max(abs(pl), abs(pr))

assert furthest_two_scenarios2("_R__LL_") == 5

# Bài 5(a)
def closest_distance(moves):
    L, R, k = moves.count("L"), moves.count("R"), moves.count("_")
    d = R - L
    return min(abs(d - k + 2 * a) for a in range(k + 1))

# Bài 5(b)
def reachable_positions(moves):
    L, R, k = moves.count("L"), moves.count("R"), moves.count("_")
    d = R - L
    return [d - k + 2 * a for a in range(k + 1)]

assert reachable_positions("L_RL__R") == [-3, -1, 1, 3]
assert closest_distance("L_RL__R") == 1
assert closest_distance("LR") == 0

random.seed(11)
for _ in range(300):
    moves = "".join(random.choice("LR_") for _ in range(random.randint(1, 10)))
    pos = {0}
    for ch in moves:
        pos = {p - 1 for p in pos} if ch == "L" else {p + 1 for p in pos} if ch == "R" else {p - 1 for p in pos} | {p + 1 for p in pos}
    assert sorted(pos) == reachable_positions(moves)
    assert closest_distance(moves) == min(abs(p) for p in pos)
    assert furthest_todo_ans(moves) == furthest_two_scenarios2(moves) == furthest_bruteforce(moves)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ô tự do + cực đại ⇒ dồn về một phía; bước bắt buộc triệt tiêu)

```python
class Solution:
    def furthestDistanceFromOrigin(self, moves: str) -> int:
        return abs(moves.count("L") - moves.count("R")) + moves.count("_")
```

**Checklist**

- [ ] Hiểu `'_'` là **một bước đi** (trái hoặc phải tuỳ chọn), không phải đứng yên?
- [ ] Các bước `L`, `R` bắt buộc **triệt tiêu** nhau (`|L − R|`)?
- [ ] Dồn mọi `_` về **cùng một phía** (không chia đều)?
- [ ] Có `abs` (trả khoảng cách, không phải vị trí có dấu)?
- [ ] Đã thử: không có `_`, toàn `_`, `L` và `R` cân bằng, một ký tự?
- [ ] Đã so công thức với brute force (`product`) trên chuỗi nhỏ?
- [ ] Signature đúng `furthestDistanceFromOrigin(self, moves: str) -> int`?